Bloco D — Métricas do Olist (passos 14 a 19, 20 min)
Seis consultas no SQL Editor, uma por passo. Salve cada uma (botão Save) com o nome do
passo, por exemplo "aula2_passo14_pedidos_mes": você vai reutilizá-las nas aulas 3 e 4.
[VERIFICAR EM SALA: nome do botão de salvar consulta]
O Databricks pode omitir zeros finais nos números (848860.1 em vez de 848860.10). Isso
não é diferença.
14. Pedidos por mês


In [0]:
%sql
SELECT
DATE(DATE_TRUNC('MONTH', order_purchase_timestamp)) AS mes,
COUNT(*)                 AS pedidos,                           
COUNT_IF(order_status = 'delivered')   AS pedidos_entregues             
FROM workspace.bronze.olist_orders
GROUP BY 1
ORDER BY 1;

mes,pedidos,pedidos_entregues
2016-09-01,4,1
2016-10-01,324,265
2016-12-01,1,1
2017-01-01,800,750
2017-02-01,1780,1653
2017-03-01,2682,2546
2017-04-01,2404,2303
2017-05-01,3700,3546
2017-06-01,3245,3135
2017-07-01,4026,3872


15. Receita de produtos, frete e GMV por mês, de dois jeitos

In [0]:
%sql
WITH itens AS (
SELECT order_id, SUM(price) AS produtos, SUM(freight_value) AS frete
FROM workspace.bronze.olist_order_items
GROUP BY order_id
),
pagamentos AS (
SELECT order_id, SUM(payment_value) AS pago
FROM workspace.bronze.olist_order_payments
GROUP BY order_id
)
SELECT
DATE(DATE_TRUNC('MONTH', o.order_purchase_timestamp)) AS mes,
ROUND(SUM(i.produtos), 2)   AS receita_produtos,        
ROUND(SUM(i.frete), 2)     AS frete,
ROUND(SUM(i.produtos + i.frete), 2) AS gmv_itens,
ROUND(SUM(p.pago), 2)   AS gmv_pagamentos
FROM workspace.bronze.olist_orders o
JOIN itens i           
ON i.order_id = o.order_id
LEFT JOIN pagamentos p ON p.order_id = o.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY 1
ORDER BY 1;

mes,receita_produtos,frete,gmv_itens,gmv_pagamentos
2016-09-01,207.86,71.83,279.69,136.23
2016-10-01,44507.3,6847.22,51354.52,51430.57
2016-12-01,10.9,8.72,19.62,19.62
2017-01-01,120098.27,16845.19,136943.46,137006.76
2017-02-01,244959.35,38602.34,283561.69,283621.94
2017-03-01,368341.32,57276.64,425617.96,425656.4
2017-04-01,353842.98,52005.63,405848.61,405988.38
2017-05-01,503159.19,79551.64,582710.83,582926.16
2017-06-01,429916.61,69735.63,499652.24,499827.47
2017-07-01,492287.3,86466.43,578753.73,578858.58


16. Ticket médio e ticket mediano dos pedidos entregues.

In [0]:
%sql
WITH por_pedido AS (
SELECT i.order_id, SUM(i.price + i.freight_value) AS valor
FROM workspace.bronze.olist_order_items i
JOIN workspace.bronze.olist_orders o ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY i.order_id
)
SELECT
COUNT(*)  AS pedidos,              
ROUND(AVG(valor), 2)    AS ticket_medio,
ROUND(MEDIAN(valor), 2) AS ticket_mediano
FROM por_pedido;


pedidos,ticket_medio,ticket_mediano
96478,159.83,105.28


17. Pedidos por cliente e taxa de recompra, contando de dois jeitos

In [0]:
%sql
WITH pedidos AS (
SELECT o.order_id, o.customer_id, c.customer_unique_id
FROM workspace.bronze.olist_orders o
JOIN workspace.bronze.olist_customers c ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
),
por_cliente_real AS (
SELECT customer_unique_id AS cliente, COUNT(*) AS n FROM pedidos GROUP BY 
customer_unique_id
),
por_customer_id AS (
SELECT customer_id AS cliente, COUNT(*) AS n FROM pedidos GROUP BY 
customer_id
)
SELECT 'customer_unique_id' AS contando_por,
COUNT(*) AS clientes,
ROUND(AVG(n), 3) AS pedidos_por_cliente,
ROUND(100 * AVG(CASE WHEN n >= 2 THEN 1 ELSE 0 END), 2) AS 
taxa_recompra_pct
FROM por_cliente_real
UNION ALL
SELECT 'customer_id',
COUNT(*),
ROUND(AVG(n), 3),
ROUND(100 * AVG(CASE WHEN n >= 2 THEN 1 ELSE 0 END), 2)
FROM por_customer_id;

contando_por,clientes,pedidos_por_cliente,taxa_recompra_pct
customer_unique_id,93358,1.033,3.0
customer_id,96478,1.0,0.0


18. Entregas atrasadas, por mês e por estado.


In [0]:
%sql
SELECT
DATE(DATE_TRUNC('MONTH', order_purchase_timestamp)) AS mes,
COUNT(*) AS entregues,
ROUND(100 * AVG(CASE WHEN DATE(order_delivered_customer_date) > 
DATE(order_estimated_delivery_date)
THEN 1 ELSE 0 END), 2) AS pct_atrasados,
ROUND(AVG(CASE WHEN DATE(order_delivered_customer_date) > 
DATE(order_estimated_delivery_date)
THEN DATEDIFF(DATE(order_delivered_customer_date), 
DATE(order_estimated_delivery_date))
END), 1) AS dias_medios_de_atraso
FROM workspace.bronze.olist_orders
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
GROUP BY 1
ORDER BY 1;

mes,entregues,pct_atrasados,dias_medios_de_atraso
2016-09-01,1,100.0,36.0
2016-10-01,265,0.75,9.5
2016-12-01,1,0.0,null
2017-01-01,750,2.93,20.2
2017-02-01,1653,2.96,20.8
2017-03-01,2546,4.56,25.4
2017-04-01,2303,6.56,12.3
2017-05-01,3545,2.99,12.5
2017-06-01,3135,3.03,14.3
2017-07-01,3872,2.79,11.7


In [0]:
%sql
SELECT
c.customer_state AS uf,
COUNT(*) AS entregues,
ROUND(100 * AVG(CASE WHEN DATE(o.order_delivered_customer_date) > 
DATE(o.order_estimated_delivery_date)
THEN 1 ELSE 0 END), 2) AS pct_atrasados
FROM workspace.bronze.olist_orders o
JOIN workspace.bronze.olist_customers c ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY pct_atrasados DESC;

uf,entregues,pct_atrasados
AL,397,21.41
MA,717,17.43
SE,335,15.22
PI,476,13.87
CE,1279,13.76
RR,41,12.2
BA,3256,12.16
RJ,12350,12.11
PA,946,11.21
ES,1995,10.73


19. Nota média por faixa de atraso.


In [0]:
%sql
WITH nota AS (
SELECT order_id, AVG(review_score) AS nota
FROM workspace.bronze.olist_order_reviews
GROUP BY order_id
),
entregas AS (
SELECT order_id,
DATEDIFF(DATE(order_delivered_customer_date), 
DATE(order_estimated_delivery_date)) AS dias_vs_prazo
FROM workspace.bronze.olist_orders
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
)
SELECT
CASE WHEN e.dias_vs_prazo <= 0  THEN '1. no prazo ou antes'
WHEN e.dias_vs_prazo <= 3  THEN '2. 1 a 3 dias'
WHEN e.dias_vs_prazo <= 7  THEN '3. 4 a 7 dias'
WHEN e.dias_vs_prazo <= 14 THEN '4. 8 a 14 dias'
ELSE '5. mais de 14 dias' END AS faixa_de_atraso,
COUNT(*)              
AS pedidos,
ROUND(AVG(n.nota), 2) AS nota_media
FROM entregas e
JOIN nota n ON n.order_id = e.order_id
GROUP BY 1
ORDER BY 1;

faixa_de_atraso,pedidos,nota_media
1. no prazo ou antes,89443,4.29
2. 1 a 3 dias,1852,3.29
3. 4 a 7 dias,1748,2.11
4. 8 a 14 dias,1446,1.67
5. mais de 14 dias,1335,1.73


Bloco E — Métricas da Prateleira (passos 20 a 25, 20 min)
Só comece este bloco depois que o ponto de checagem C deu 
30/09/2026, o último dia com dados.
20. DAU, WAU e MAU em 30/09/2026.

In [0]:
%sql
SELECT
COUNT(DISTINCT CASE WHEN evento_data = DATE'2026-09-30'  THEN usuario_id 
END) AS dau,
COUNT(DISTINCT CASE WHEN evento_data >= DATE'2026-09-24' THEN usuario_id 
END) AS wau,
COUNT(DISTINCT usuario_id)                                                    
AS mau
FROM workspace.bronze.saas_eventos
WHERE evento_data BETWEEN DATE'2026-09-01' AND DATE'2026-09-30';

dau,wau,mau
2052,3362,3643


21. Stickiness mensal.


In [0]:
%sql
WITH dau AS (
SELECT evento_data AS dia, COUNT(DISTINCT usuario_id) AS dau
FROM workspace.bronze.saas_eventos
GROUP BY evento_data
),
mau AS (
SELECT DATE(DATE_TRUNC('MONTH', evento_data)) AS mes, COUNT(DISTINCT 
usuario_id) AS mau
FROM workspace.bronze.saas_eventos
GROUP BY 1
)
SELECT
m.mes,
ROUND(AVG(d.dau), 1)         
AS dau_medio,
m.mau,
ROUND(AVG(d.dau) / m.mau, 3) AS stickiness
FROM mau m
JOIN dau d ON DATE(DATE_TRUNC('MONTH', d.dia)) = m.mes
GROUP BY m.mes, m.mau
ORDER BY m.mes;


mes,dau_medio,mau,stickiness
2024-10-01,24.7,133,0.186
2024-11-01,71.7,276,0.26
2024-12-01,125.1,366,0.342
2025-01-01,184.1,483,0.381
2025-02-01,238.0,588,0.405
2025-03-01,300.4,721,0.417
2025-04-01,355.2,807,0.44
2025-05-01,430.1,983,0.438
2025-06-01,490.6,1118,0.439
2025-07-01,560.3,1211,0.463


Bônus — curva L28 (só se sobrar tempo):


In [0]:
WITH dias AS (
SELECT usuario_id, COUNT(DISTINCT evento_data) AS dias_ativos
FROM workspace.bronze.saas_eventos
WHERE evento_data BETWEEN DATE'2026-09-03' AND DATE'2026-09-30'
GROUP BY usuario_id
)
SELECT dias_ativos, COUNT(*) AS usuarios
FROM dias
GROUP BY dias_ativos
ORDER BY dias_ativos;

dias_ativos,usuarios
1,95
2,86
3,80
4,72
5,56
6,78
7,95
8,127
9,164
10,199


22. MRR e contas pagantes no fim de cada mês.

In [0]:
WITH meses AS (
SELECT explode(sequence(DATE'2024-10-01', DATE'2026-09-01', INTERVAL 1 
MONTH)) AS mes
)
SELECT
m.mes,
COUNT(DISTINCT a.conta_id) AS contas_pagantes,
COALESCE(SUM(a.mrr), 0)    AS mrr
FROM meses m
LEFT JOIN workspace.bronze.saas_assinaturas a
ON a.inicio <= LAST_DAY(m.mes)
AND (a.fim IS NULL OR a.fim > LAST_DAY(m.mes))
GROUP BY m.mes
ORDER BY m.mes;

mes,contas_pagantes,mrr
2024-10-01,27,3563.00
2024-11-01,68,10412.00
2024-12-01,115,18415.00
2025-01-01,147,25383.00
2025-02-01,177,31933.00
2025-03-01,215,38555.00
2025-04-01,245,44025.00
2025-05-01,293,54557.00
2025-06-01,340,62800.00
2025-07-01,386,72034.00


23. Movimentos de MRR por mês.


In [0]:
WITH meses AS (
SELECT explode(sequence(DATE'2024-10-01', DATE'2026-09-01', INTERVAL 1 
MONTH)) AS mes
),
contas AS (
  SELECT DISTINCT conta_id FROM workspace.bronze.saas_assinaturas
),
conta_mes AS (
  SELECT m.mes, c.conta_id, COALESCE(SUM(a.mrr), 0) AS mrr
  FROM meses m
  CROSS JOIN contas c
  LEFT JOIN workspace.bronze.saas_assinaturas a
    ON a.conta_id = c.conta_id
   AND a.inicio <= LAST_DAY(m.mes)
   AND (a.fim IS NULL OR a.fim > LAST_DAY(m.mes))
  GROUP BY m.mes, c.conta_id
),
com_anterior AS (
  SELECT mes, conta_id, mrr,
         LAG(mrr, 1, 0) OVER (PARTITION BY conta_id ORDER BY mes) AS 
mrr_anterior,
         COALESCE(MAX(mrr) OVER (PARTITION BY conta_id ORDER BY mes
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND 1 
PRECEDING), 0) AS maior_mrr_antes
  FROM conta_mes
)
SELECT
  mes,
  SUM(CASE WHEN mrr_anterior = 0 AND mrr > 0 AND maior_mrr_antes = 0 THEN mrr 
ELSE 0 END)    AS new_mrr,
  SUM(CASE WHEN mrr_anterior = 0 AND mrr > 0 AND maior_mrr_antes > 0 THEN mrr 
ELSE 0 END)    AS reativacao_mrr,
  SUM(CASE WHEN mrr_anterior > 0 AND mrr > mrr_anterior THEN mrr - 
mrr_anterior ELSE 0 END)  AS expansion_mrr,
  -SUM(CASE WHEN mrr_anterior > 0 AND mrr > 0 AND mrr < mrr_anterior
            THEN mrr_anterior - mrr ELSE 0 END)                                 
AS contraction_mrr,
  -SUM(CASE WHEN mrr_anterior > 0 AND mrr = 0 THEN mrr_anterior ELSE 0 END)     
AS churn_mrr,
  SUM(mrr)                                                                      
AS mrr_fim_do_mes
FROM com_anterior
GROUP BY mes
ORDER BY mes;

mes,new_mrr,reativacao_mrr,expansion_mrr,contraction_mrr,churn_mrr,mrr_fim_do_mes
2024-10-01,3563.00,0.00,0.00,0.00,0.00,3563.00
2024-11-01,7127.00,0.00,0.00,0.00,-278.00,10412.00
2024-12-01,8270.00,0.00,200.00,-100.00,-367.00,18415.00
2025-01-01,7602.00,0.00,100.00,-100.00,-634.00,25383.00
2025-02-01,6746.00,0.00,360.00,0.00,-556.00,31933.00
2025-03-01,6700.00,0.00,100.00,0.00,-178.00,38555.00
2025-04-01,5726.00,0.00,300.00,0.00,-556.00,44025.00
2025-05-01,10477.00,0.00,860.00,-260.00,-545.00,54557.00
2025-06-01,9066.00,0.00,300.00,0.00,-1123.00,62800.00
2025-07-01,10762.00,0.00,560.00,-100.00,-1988.00,72034.00


24. Churn mensal de contas e de receita.


In [0]:
WITH meses AS (
  SELECT explode(sequence(DATE'2024-10-01', DATE'2026-09-01', INTERVAL 1 
MONTH)) AS mes
),
contas AS (
  SELECT DISTINCT conta_id FROM workspace.bronze.saas_assinaturas
),
conta_mes AS (
  SELECT m.mes, c.conta_id, COALESCE(SUM(a.mrr), 0) AS mrr
  FROM meses m
  CROSS JOIN contas c
  LEFT JOIN workspace.bronze.saas_assinaturas a
    ON a.conta_id = c.conta_id
   AND a.inicio <= LAST_DAY(m.mes)
   AND (a.fim IS NULL OR a.fim > LAST_DAY(m.mes))
  GROUP BY m.mes, c.conta_id
),
com_anterior AS (
  SELECT mes, conta_id, mrr,
         LAG(mrr, 1, 0) OVER (PARTITION BY conta_id ORDER BY mes) AS 
mrr_anterior
  FROM conta_mes
)
SELECT
  mes,
  COUNT_IF(mrr_anterior > 0)             AS contas_no_inicio,
  COUNT_IF(mrr_anterior > 0 AND mrr = 0) AS contas_canceladas,
  ROUND(100 * COUNT_IF(mrr_anterior > 0 AND mrr = 0) / 
NULLIF(COUNT_IF(mrr_anterior > 0), 0), 2) AS churn_contas_pct,
  ROUND(100 * SUM(CASE WHEN mrr_anterior > 0 AND mrr = 0 THEN mrr_anterior 
ELSE 0 END)
            / NULLIF(SUM(mrr_anterior), 0), 2) AS churn_receita_pct
FROM com_anterior
GROUP BY mes
HAVING COUNT_IF(mrr_anterior > 0) > 0
ORDER BY mes;

mes,contas_no_inicio,contas_canceladas,churn_contas_pct,churn_receita_pct
2024-11-01,27,2,7.41,7.80
2024-12-01,68,3,4.41,3.52
2025-01-01,115,6,5.22,3.44
2025-02-01,147,4,2.72,2.19
2025-03-01,177,2,1.13,0.56
2025-04-01,215,4,1.86,1.44
2025-05-01,245,5,2.04,1.24
2025-06-01,293,7,2.39,2.06
2025-07-01,340,12,3.53,3.17
2025-08-01,386,16,4.15,3.39


25. Primeira visão de coorte.


In [0]:
WITH meses AS (
SELECT explode(sequence(DATE'2024-10-01', DATE'2026-09-01', INTERVAL 1 
MONTH)) AS mes
),
contas AS (
SELECT DISTINCT conta_id FROM workspace.bronze.saas_assinaturas
),
conta_mes AS (
SELECT m.mes, c.conta_id, COALESCE(SUM(a.mrr), 0) AS mrr
FROM meses m
CROSS JOIN contas c
LEFT JOIN workspace.bronze.saas_assinaturas a
ON a.conta_id = c.conta_id
AND a.inicio <= LAST_DAY(m.mes)
AND (a.fim IS NULL OR a.fim > LAST_DAY(m.mes))
GROUP BY m.mes, c.conta_id
),
coorte AS (
SELECT conta_id, DATE(DATE_TRUNC('MONTH', MIN(inicio))) AS mes_coorte
FROM workspace.bronze.saas_assinaturas
GROUP BY conta_id
),
base AS (
SELECT k.mes_coorte, cm.conta_id,
CAST(months_between(cm.mes, k.mes_coorte) AS INT) AS m,
CASE WHEN cm.mrr > 0 THEN 1 ELSE 0 END AS pagante
FROM conta_mes cm
JOIN coorte k ON k.conta_id = cm.conta_id
WHERE cm.mes >= k.mes_coorte
)
SELECT
mes_coorte,
COUNT(DISTINCT conta_id) AS contas,
ROUND(100 * AVG(CASE WHEN m = 0  THEN pagante END), 1) AS m0,
ROUND(100 * AVG(CASE WHEN m = 1  THEN pagante END), 1) AS m1,
ROUND(100 * AVG(CASE WHEN m = 2  THEN pagante END), 1) AS m2,
ROUND(100 * AVG(CASE WHEN m = 3  THEN pagante END), 1) AS m3,
ROUND(100 * AVG(CASE WHEN m = 6  THEN pagante END), 1) AS m6,
ROUND(100 * AVG(CASE WHEN m = 12 THEN pagante END), 1) AS m12
FROM base
GROUP BY mes_coorte
ORDER BY mes_coorte;

mes_coorte,contas,m0,m1,m2,m3,m6,m12
2024-10-01,27,100.0,92.6,88.9,88.9,85.2,63.0
2024-11-01,44,97.7,93.2,90.9,86.4,81.8,70.5
2024-12-01,50,100.0,90.0,88.0,88.0,84.0,74.0
2025-01-01,38,100.0,97.4,97.4,94.7,92.1,81.6
2025-02-01,34,100.0,97.1,97.1,94.1,79.4,64.7
2025-03-01,40,100.0,95.0,95.0,95.0,92.5,82.5
2025-04-01,34,100.0,100.0,97.1,94.1,91.2,70.6
2025-05-01,53,100.0,96.2,94.3,92.5,90.6,62.3
2025-06-01,56,96.4,89.3,83.9,78.6,71.4,57.1
2025-07-01,58,100.0,94.8,91.4,86.2,79.3,67.2


Bloco F — Exercício de ambiguidade (passos 26 a 28, 15 min)
Metade da turma calcula "clientes ativos" do Olist com a definição A, metade com a B. O
professor diz qual metade é a sua (por exemplo, número de chamada par ou ímpar). O
"hoje" do exercício é 31/08/2018, o último mês completo do dataset.
26. Calcule os clientes ativos com a definição da sua metade da turma.
Definição A: cliente que comprou nos 90 dias até 31/08/2018, com pedido em qualquer
status

In [0]:
SELECT COUNT(DISTINCT c.customer_unique_id) AS clientes_ativos
FROM workspace.bronze.olist_orders o
JOIN workspace.bronze.olist_customers c ON c.customer_id = o.customer_id
WHERE o.order_purchase_timestamp >= DATE'2018-06-03'
AND o.order_purchase_timestamp <  DATE'2018-09-01';

clientes_ativos
18404


Definição B: cliente com pedido entregue, comprado nos 30 dias até 31/08/2018.

In [0]:
SELECT COUNT(DISTINCT c.customer_unique_id) AS clientes_ativos
FROM workspace.bronze.olist_orders o
JOIN workspace.bronze.olist_customers c ON c.customer_id = o.customer_id
WHERE o.order_purchase_timestamp >= DATE'2018-08-02'
AND o.order_purchase_timestamp <  DATE'2018-09-01'
AND o.order_status = 'delivered';

clientes_ativos
6008


In [0]:
27. Registre e discuta.
1. Registre o seu número no formulário indicado pelo professor.
2. Quando os dois números aparecerem no telão, responda com o colega ao lado, por
escrito:
Qual dos dois números você levaria ao CEO como "clientes ativos"? Por quê?
Qual das seis perguntas do slide 36 (o que conta, janela, grão, filtros, fonte, dono)
explica a diferença?
Reescreva a definição que você escolheu respondendo às seis perguntas, em até
três linhas.
28. Bônus: "conta ativa" na Prateleira.
Rode a consulta abaixo, que calcula duas definições de conta ativa em 30/09/2026: ter
assinatura paga e ter usado o produto nos últimos 28 dias.

In [0]:
SELECT
(SELECT COUNT(DISTINCT conta_id)
FROM workspace.bronze.saas_assinaturas
WHERE inicio <= DATE'2026-09-30' AND (fim IS NULL OR fim > DATE'2026-09-30')) AS contas_pagantes,
(SELECT COUNT(DISTINCT conta_id)
FROM workspace.bronze.saas_eventos
WHERE evento_data BETWEEN DATE'2026-09-03' AND DATE'2026-09-30')            
AS contas_com_uso_28_dias;

contas_pagantes,contas_com_uso_28_dias
1093,1386
